# Paper-ready data outputs

This notebook creates the descriptive outputs needed for the course paper **without
rerunning the cleaning pipeline or refitting the demand models**.

It reuses the saved outputs from:

- `01_sample_creation.ipynb`
- `02_demand_model_selection.ipynb`

The notebook produces:

1. a compact product-level paper table;
2. final-sample summary statistics;
3. promotion-episode and promotion-depth support tables;
4. a two-panel figure showing promotion-depth support and sales relative to a rolling-PPML no-promotion baseline around
   isolated promotion episodes;
5. CSV and LaTeX versions of the paper tables.

The first run creates a small cached Parquet file containing only the selected
eight products, fifteen stores, and eligible store-product panels. Later runs reuse
that cache unless the upstream selection files or source Parquet file change.

## 1. Imports and configuration

In [10]:
from __future__ import annotations

from hashlib import sha256
from pathlib import Path
import json
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from statsmodels.iolib.smpickle import load_pickle

warnings.filterwarnings("ignore", category=FutureWarning)

# Locate the repository from either the repository root or notebooks/.
CURRENT_DIR = Path.cwd().resolve()
POSSIBLE_ROOTS = [CURRENT_DIR, CURRENT_DIR.parent]

PROJECT_ROOT = next(
    (
        root
        for root in POSSIBLE_ROOTS
        if (root / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from the "
        "repository root or the notebooks directory."
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_DATA_PATH = PROCESSED_DIR / "cereal_demand_model_data.parquet"
PRODUCT_LOOKUP_PATH = PROCESSED_DIR / "cereal_product_lookup.parquet"

SELECTED_PRODUCTS_PATH = TABLE_DIR / "selected_products.csv"
STORE_SELECTION_PATH = TABLE_DIR / "store_selection_table.csv"
PANEL_SELECTION_PATH = TABLE_DIR / "panel_selection_table.csv"
PRODUCT_SELECTION_PATH = TABLE_DIR / "product_selection_table.csv"

MODEL_DIR = PROJECT_ROOT / "results" / "models"
ROLLING_MODEL_DIR = MODEL_DIR / "product_promotion_rolling_ppml"
ROLLING_MANIFEST_PATH = (
    MODEL_DIR / "product_promotion_rolling_ppml_manifest.pkl"
)
PREDICTION_CONTEXT_PATH = (
    PROCESSED_DIR / "product_promotion_prediction_context.pkl"
)

CACHE_PATH = PROCESSED_DIR / "paper_selected_sample.parquet"
CACHE_MANIFEST_PATH = PROCESSED_DIR / "paper_selected_sample_manifest.json"

N_STORES = 15
TRAIN_SHARE = 0.60
CALIBRATION_SHARE = 0.20

# Historical information used to construct promotion-depth support.
# The test period is intentionally excluded.
SUPPORT_SPLITS = ("train", "calibration")
DEPTH_CLUSTER_WIDTH = 0.05
MIN_SUPPORT_OBSERVATIONS = 15
MIN_SUPPORT_PANELS = 3
MAX_POSITIVE_DEPTHS = 3

# Fallback selection if no downstream final action-grid file is available.
# "highest_support" keeps the best-supported eligible clusters.
ACTION_SELECTION_RULE = "highest_support"

# Event-study settings.
PRE_EVENT_WEEKS = 4
POST_EVENT_WEEKS = 6
REQUIRE_SINGLE_WEEK_PROMOTION = True
REQUIRE_NO_OTHER_PROMOTION_IN_WINDOW = True
MIN_PRE_PERIOD_MEAN_SALES = 1e-8
EVENT_BOOTSTRAP_REPLICATIONS = 2_000
EVENT_BOOTSTRAP_SEED = 20260723

# Cache controls.
USE_CACHE = True
FORCE_REBUILD_CACHE = False

required_paths = [
    SOURCE_DATA_PATH,
    PRODUCT_LOOKUP_PATH,
    SELECTED_PRODUCTS_PATH,
    STORE_SELECTION_PATH,
    PANEL_SELECTION_PATH,
    PRODUCT_SELECTION_PATH,
    ROLLING_MANIFEST_PATH,
    PREDICTION_CONTEXT_PATH,
]

missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    missing_text = "\n".join(f"  - {path}" for path in missing_paths)
    raise FileNotFoundError(
        "Required saved outputs are missing. Run notebooks 01 and 02 once, "
        f"then rerun this notebook:\n{missing_text}"
    )

print("Project root:", PROJECT_ROOT)
print("Source data:", SOURCE_DATA_PATH)
print("Output tables:", TABLE_DIR)
print("Output figures:", FIGURE_DIR)

Project root: C:\Users\janza\price-of-extrapolation
Source data: C:\Users\janza\price-of-extrapolation\data\processed\cereal_demand_model_data.parquet
Output tables: C:\Users\janza\price-of-extrapolation\results\tables
Output figures: C:\Users\janza\price-of-extrapolation\results\figures


## 2. Load the saved selection decisions

In [11]:
def normalize_identifier(series: pd.Series) -> pd.Series:
    """Normalize identifiers read from CSV or Parquet to comparable strings."""
    normalized = series.astype("string").str.strip()
    # Remove a trailing '.0' introduced when integer IDs were stored as floats.
    return normalized.str.replace(r"\.0$", "", regex=True)


def parse_boolean(series: pd.Series) -> pd.Series:
    """Parse bool, numeric, or string-valued CSV indicators safely."""
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    normalized = series.astype("string").str.strip().str.lower()
    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }
    parsed = normalized.map(mapping)

    if parsed.isna().any():
        bad_values = sorted(normalized.loc[parsed.isna()].dropna().unique())
        raise ValueError(f"Could not parse boolean values: {bad_values}")

    return parsed.astype(bool)


selected_products_file = pd.read_csv(
    SELECTED_PRODUCTS_PATH,
    dtype={"upc": "string"},
)
selected_upcs = (
    normalize_identifier(selected_products_file["upc"])
    .dropna()
    .drop_duplicates()
    .tolist()
)

store_selection = pd.read_csv(
    STORE_SELECTION_PATH,
    dtype={"store": "string"},
)
store_selection["store"] = normalize_identifier(
    store_selection["store"]
)

selected_stores = (
    store_selection.loc[
        parse_boolean(store_selection["eligible"]),
        "store",
    ]
    .head(N_STORES)
    .dropna()
    .drop_duplicates()
    .tolist()
)

panel_selection = pd.read_csv(
    PANEL_SELECTION_PATH,
    dtype={"store_upc": "string"},
)
panel_selection["store_upc"] = normalize_identifier(
    panel_selection["store_upc"]
)

eligible_panels = set(
    panel_selection.loc[
        parse_boolean(panel_selection["eligible"]),
        "store_upc",
    ]
    .dropna()
    .tolist()
)

product_selection = pd.read_csv(
    PRODUCT_SELECTION_PATH,
    dtype={"upc": "string"},
)
product_selection["upc"] = normalize_identifier(
    product_selection["upc"]
)

if not selected_upcs:
    raise ValueError("No selected products were found.")
if not selected_stores:
    raise ValueError("No selected stores were found.")
if not eligible_panels:
    raise ValueError("No eligible store-product panels were found.")

print(f"Selected products: {len(selected_upcs)}")
print(f"Selected stores: {len(selected_stores)}")
print(f"Eligible panels in saved audit table: {len(eligible_panels)}")

Selected products: 8
Selected stores: 15
Eligible panels in saved audit table: 120


## 3. Load or rebuild the compact selected-sample cache

In [12]:
def coerce_values_for_arrow(
    values: list[str],
    arrow_type: pa.DataType,
) -> list:
    """Convert string identifiers to the physical type used in Parquet."""
    if pa.types.is_string(arrow_type) or pa.types.is_large_string(arrow_type):
        return [str(value) for value in values]
    if pa.types.is_integer(arrow_type):
        return [int(float(value)) for value in values]
    if pa.types.is_floating(arrow_type):
        return [float(value) for value in values]
    raise TypeError(
        f"Unsupported identifier type for filtered Parquet read: {arrow_type}"
    )


def selection_fingerprint() -> str:
    payload = {
        "source_path": str(SOURCE_DATA_PATH.resolve()),
        "source_mtime_ns": SOURCE_DATA_PATH.stat().st_mtime_ns,
        "selected_upcs": sorted(selected_upcs),
        "selected_stores": sorted(selected_stores),
        "eligible_panels": sorted(eligible_panels),
    }
    serialized = json.dumps(payload, sort_keys=True).encode("utf-8")
    return sha256(serialized).hexdigest()


expected_fingerprint = selection_fingerprint()
cache_is_current = False

if USE_CACHE and CACHE_PATH.exists() and CACHE_MANIFEST_PATH.exists():
    with CACHE_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        cache_manifest = json.load(handle)
    cache_is_current = (
        cache_manifest.get("fingerprint") == expected_fingerprint
    )

if cache_is_current and not FORCE_REBUILD_CACHE:
    selected_sample = pd.read_parquet(CACHE_PATH)
    print(f"Loaded cached selected sample: {CACHE_PATH}")
else:
    source_dataset = ds.dataset(
        SOURCE_DATA_PATH,
        format="parquet",
    )

    upc_type = source_dataset.schema.field("upc").type
    arrow_upcs = coerce_values_for_arrow(selected_upcs, upc_type)

    selected_columns = [
        "store",
        "upc",
        "store_upc",
        "week",
        "week_start",
        "week_end",
        "move",
        "model_unit_price",
        "regular_price",
        "discount_depth",
        "price_imputed",
        "pricing_state",
        "promo_state",
        "post_promo",
        "gross_margin_pct_observed",
    ]

    missing_source_columns = set(selected_columns).difference(
        source_dataset.schema.names
    )
    if missing_source_columns:
        raise ValueError(
            "Source Parquet is missing required columns: "
            f"{sorted(missing_source_columns)}"
        )

    arrow_table = source_dataset.to_table(
        columns=selected_columns,
        filter=ds.field("upc").isin(arrow_upcs),
    )
    selected_sample = arrow_table.to_pandas()
    del arrow_table

    for identifier in ["store", "upc", "store_upc"]:
        selected_sample[identifier] = normalize_identifier(
            selected_sample[identifier]
        )

    selected_sample = selected_sample.loc[
        selected_sample["store"].isin(selected_stores)
        & selected_sample["store_upc"].isin(eligible_panels)
    ].copy()

    selected_sample = (
        selected_sample
        .sort_values(["store_upc", "week"], kind="mergesort")
        .reset_index(drop=True)
    )

    selected_sample.to_parquet(
        CACHE_PATH,
        index=False,
        engine="pyarrow",
        compression="zstd",
    )

    cache_manifest = {
        "fingerprint": expected_fingerprint,
        "rows": int(len(selected_sample)),
        "products": int(selected_sample["upc"].nunique()),
        "stores": int(selected_sample["store"].nunique()),
        "panels": int(selected_sample["store_upc"].nunique()),
    }
    with CACHE_MANIFEST_PATH.open("w", encoding="utf-8") as handle:
        json.dump(cache_manifest, handle, indent=2)

    print(f"Created selected-sample cache: {CACHE_PATH}")

for identifier in ["store", "upc", "store_upc"]:
    selected_sample[identifier] = normalize_identifier(
        selected_sample[identifier]
    )

selected_sample["week"] = pd.to_numeric(
    selected_sample["week"],
    errors="raise",
).astype(int)
selected_sample["week_start"] = pd.to_datetime(
    selected_sample["week_start"],
    errors="coerce",
)
selected_sample["week_end"] = pd.to_datetime(
    selected_sample["week_end"],
    errors="coerce",
)
selected_sample["move"] = pd.to_numeric(
    selected_sample["move"],
    errors="coerce",
)
selected_sample["discount_depth"] = pd.to_numeric(
    selected_sample["discount_depth"],
    errors="coerce",
)
selected_sample["model_unit_price"] = pd.to_numeric(
    selected_sample["model_unit_price"],
    errors="coerce",
)
selected_sample["regular_price"] = pd.to_numeric(
    selected_sample["regular_price"],
    errors="coerce",
)
selected_sample["gross_margin_pct_observed"] = pd.to_numeric(
    selected_sample["gross_margin_pct_observed"],
    errors="coerce",
)
selected_sample["price_imputed"] = (
    selected_sample["price_imputed"].fillna(False).astype(bool)
)

pricing_state = (
    selected_sample["pricing_state"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)
selected_sample["promotion_indicator"] = (
    pricing_state.isin(["promotion", "promo"]).astype("int8")
)
selected_sample["post_promotion_indicator"] = (
    pricing_state.isin(
        ["post_promotion", "postpromo", "post_promo"]
    ).astype("int8")
)

if selected_sample.empty:
    raise ValueError(
        "The compact selected sample is empty. Check identifier formats "
        "in the saved selection tables."
    )

print(
    "Selected sample:",
    f"{len(selected_sample):,} rows,",
    f"{selected_sample['upc'].nunique()} products,",
    f"{selected_sample['store'].nunique()} stores,",
    f"{selected_sample['store_upc'].nunique()} panels.",
)

Loaded cached selected sample: C:\Users\janza\price-of-extrapolation\data\processed\paper_selected_sample.parquet
Selected sample: 42,285 rows, 8 products, 15 stores, 120 panels.


## 4. Reconstruct the saved chronological split

In [13]:
# Notebook 02 defines the split before product selection using all cleaned weeks.
source_dataset = ds.dataset(SOURCE_DATA_PATH, format="parquet")
all_week_table = source_dataset.to_table(columns=["week"])
all_weeks = np.sort(
    pd.unique(
        pd.to_numeric(
            all_week_table.column("week").to_pandas(),
            errors="raise",
        )
    )
)
del all_week_table

train_end_index = int(np.floor(TRAIN_SHARE * len(all_weeks)))
calibration_end_index = int(
    np.floor(
        (TRAIN_SHARE + CALIBRATION_SHARE)
        * len(all_weeks)
    )
)

train_weeks = set(all_weeks[:train_end_index])
calibration_weeks = set(
    all_weeks[train_end_index:calibration_end_index]
)
test_weeks = set(all_weeks[calibration_end_index:])

selected_sample["split"] = np.select(
    [
        selected_sample["week"].isin(train_weeks),
        selected_sample["week"].isin(calibration_weeks),
        selected_sample["week"].isin(test_weeks),
    ],
    ["train", "calibration", "test"],
    default="unassigned",
)

if (selected_sample["split"] == "unassigned").any():
    raise ValueError("Some selected-sample weeks could not be assigned.")

split_summary = (
    selected_sample.groupby("split", observed=True)
    .agg(
        rows=("move", "size"),
        weeks=("week", "nunique"),
        products=("upc", "nunique"),
        stores=("store", "nunique"),
        panels=("store_upc", "nunique"),
        start_date=("week_start", "min"),
        end_date=("week_end", "max"),
    )
    .reindex(["train", "calibration", "test"])
    .reset_index()
)

display(split_summary)

,split,rows,weeks,products,stores,panels,start_date,end_date
0,train,25006,214,8,15,120,1989-10-12,1993-11-17
1,calibration,8639,72,8,15,120,1993-11-25,1995-12-06
2,test,8640,72,8,15,120,1995-12-07,1997-05-07


## 5. Identify promotion episodes

In [14]:
episode_data = (
    selected_sample
    .sort_values(["store_upc", "week"], kind="mergesort")
    .copy()
)

grouped_panels = episode_data.groupby(
    "store_upc",
    observed=True,
    sort=False,
)

previous_week = grouped_panels["week"].shift(1)
previous_promotion = (
    grouped_panels["promotion_indicator"].shift(1).fillna(0)
)

consecutive_previous_week = (
    episode_data["week"] - previous_week
).eq(1)

episode_data["promotion_start"] = (
    episode_data["promotion_indicator"].eq(1)
    & ~(
        previous_promotion.eq(1)
        & consecutive_previous_week
    )
)

episode_data["episode_number"] = (
    episode_data["promotion_start"]
    .groupby(episode_data["store_upc"], observed=True)
    .cumsum()
    .astype(int)
)

promotion_rows = episode_data.loc[
    episode_data["promotion_indicator"].eq(1)
].copy()

promotion_episodes = (
    promotion_rows.groupby(
        ["store_upc", "episode_number"],
        observed=True,
    )
    .agg(
        store=("store", "first"),
        upc=("upc", "first"),
        start_week=("week", "min"),
        end_week=("week", "max"),
        start_date=("week_start", "min"),
        end_date=("week_end", "max"),
        promotion_weeks=("week", "nunique"),
        mean_depth=("discount_depth", "mean"),
        median_depth=("discount_depth", "median"),
        max_depth=("discount_depth", "max"),
    )
    .reset_index()
)

promotion_episodes["episode_length"] = (
    promotion_episodes["end_week"]
    - promotion_episodes["start_week"]
    + 1
)

# Consecutive episodes should have the same calendar and row length.
promotion_episodes["consecutive_episode"] = (
    promotion_episodes["promotion_weeks"]
    == promotion_episodes["episode_length"]
)

print(
    f"Promotion episodes: {len(promotion_episodes):,} "
    f"across {promotion_episodes['upc'].nunique()} products."
)
display(
    promotion_episodes.groupby("upc", observed=True)
    .agg(
        episodes=("episode_number", "size"),
        single_week_episodes=(
            "promotion_weeks",
            lambda x: int((x == 1).sum()),
        ),
        median_depth=("median_depth", "median"),
    )
    .reset_index()
)

Promotion episodes: 2,651 across 8 products.


,upc,episodes,single_week_episodes,median_depth
0,1600066510,232,143,0.248996
1,1600066590,440,278,0.245768
2,1600066610,465,289,0.201863
3,3800000110,93,72,0.074419
4,3800000120,442,233,0.109728
5,3800000127,352,222,0.182786
6,3800001520,404,231,0.076056
7,3800001611,223,159,0.107463


## 6. Construct product-specific promotion-depth support

In [15]:
support_source = selected_sample.loc[
    selected_sample["split"].isin(SUPPORT_SPLITS)
    & selected_sample["promotion_indicator"].eq(1)
    & selected_sample["discount_depth"].gt(0)
].copy()

support_source["depth_cluster"] = (
    (
        support_source["discount_depth"]
        / DEPTH_CLUSTER_WIDTH
    )
    .round()
    .mul(DEPTH_CLUSTER_WIDTH)
    .clip(lower=DEPTH_CLUSTER_WIDTH, upper=0.95)
    .round(4)
)

depth_support = (
    support_source.groupby(
        ["upc", "depth_cluster"],
        observed=True,
    )
    .agg(
        observations=("move", "size"),
        panels=("store_upc", "nunique"),
        stores=("store", "nunique"),
        median_observed_depth=("discount_depth", "median"),
        mean_observed_depth=("discount_depth", "mean"),
    )
    .reset_index()
)

depth_support["supported"] = (
    depth_support["observations"].ge(MIN_SUPPORT_OBSERVATIONS)
    & depth_support["panels"].ge(MIN_SUPPORT_PANELS)
)

supported_candidates = depth_support.loc[
    depth_support["supported"]
].copy()

if ACTION_SELECTION_RULE != "highest_support":
    raise ValueError(
        "Only ACTION_SELECTION_RULE='highest_support' is implemented "
        "as the transparent fallback in this reporting notebook."
    )

supported_candidates = (
    supported_candidates
    .sort_values(
        ["upc", "observations", "panels", "depth_cluster"],
        ascending=[True, False, False, True],
    )
    .groupby("upc", observed=True)
    .head(MAX_POSITIVE_DEPTHS)
    .sort_values(["upc", "depth_cluster"])
    .reset_index(drop=True)
)

supported_candidates["admissible"] = True

def format_depths(values: pd.Series) -> str:
    unique_values = sorted(pd.unique(values.dropna()))
    if not unique_values:
        return "None"
    return ", ".join(f"{100 * value:.0f}\\%" for value in unique_values)

admissible_depth_strings = (
    supported_candidates.groupby("upc", observed=True)[
        "depth_cluster"
    ]
    .apply(format_depths)
    .rename("supported_depths")
    .reset_index()
)

display(
    depth_support.sort_values(
        ["upc", "depth_cluster"]
    )
)

,upc,depth_cluster,observations,panels,stores,median_observed_depth,mean_observed_depth,supported
0,1600066510,0.05,150,15,15,0.058577,0.050599,True
1,1600066510,0.10,65,13,13,0.087500,0.090710,True
2,1600066510,0.15,8,6,6,0.135703,0.139432,False
3,1600066510,0.20,4,4,4,0.189978,0.192298,False
4,1600066510,0.25,73,15,15,0.251046,0.251459,True
...,...,...,...,...,...,...,...,...
76,3800001611,0.20,5,5,5,0.218430,0.211358,False
77,3800001611,0.25,12,10,10,0.268371,0.263158,False
78,3800001611,0.30,7,6,6,0.303951,0.297773,False
79,3800001611,0.35,11,10,10,0.368254,0.360408,False


## 7. Build isolated-event windows and the residualized event path

In [ ]:
relative_weeks = np.arange(
    -PRE_EVENT_WEEKS,
    POST_EVENT_WEEKS + 1,
)

sample_indexed = (
    selected_sample
    .set_index(["store_upc", "week"])
    .sort_index()
)

event_window_rows = []
event_metadata_rows = []

candidate_episodes = promotion_episodes.loc[
    promotion_episodes["consecutive_episode"]
].copy()

if REQUIRE_SINGLE_WEEK_PROMOTION:
    candidate_episodes = candidate_episodes.loc[
        candidate_episodes["promotion_weeks"].eq(1)
    ]

for event_id, event in candidate_episodes.reset_index(
    drop=True
).iterrows():
    panel = event["store_upc"]
    event_week = int(event["start_week"])
    desired_weeks = event_week + relative_weeks

    try:
        window = (
            sample_indexed.loc[(panel, desired_weeks), :]
            .reset_index()
        )
    except KeyError:
        continue

    # A complete window must contain every requested week exactly once.
    if len(window) != len(relative_weeks):
        continue

    window = window.sort_values("week").copy()
    window["relative_week"] = window["week"] - event_week

    if not np.array_equal(
        window["relative_week"].to_numpy(),
        relative_weeks,
    ):
        continue

    if REQUIRE_NO_OTHER_PROMOTION_IN_WINDOW:
        other_promotion = (
            window["promotion_indicator"].eq(1)
            & window["relative_week"].ne(0)
        )
        if other_promotion.any():
            continue

    pre_period = window.loc[
        window["relative_week"].between(
            -PRE_EVENT_WEEKS,
            -1,
        ),
        "move",
    ]

    if len(pre_period) != PRE_EVENT_WEEKS:
        continue

    raw_baseline_sales = pre_period.mean()
    if (
        not np.isfinite(raw_baseline_sales)
        or raw_baseline_sales <= MIN_PRE_PERIOD_MEAN_SALES
    ):
        continue

    window["event_id"] = event_id
    window["raw_baseline_sales"] = raw_baseline_sales
    window["raw_sales_index"] = (
        100.0 * window["move"] / raw_baseline_sales
    )

    event_window_rows.append(
        window[
            [
                "event_id",
                "store_upc",
                "store",
                "upc",
                "week",
                "relative_week",
                "move",
                "model_unit_price",
                "regular_price",
                "price_imputed",
                "raw_baseline_sales",
                "raw_sales_index",
                "discount_depth",
                "promotion_indicator",
                "post_promotion_indicator",
                "split",
            ]
        ]
    )

    event_metadata_rows.append(
        {
            "event_id": event_id,
            "store_upc": panel,
            "store": event["store"],
            "upc": event["upc"],
            "event_week": event_week,
            "event_date": event["start_date"],
            "promotion_depth": event["median_depth"],
            "split": window.loc[
                window["relative_week"].eq(0),
                "split",
            ].iloc[0],
        }
    )

if not event_window_rows:
    raise RuntimeError(
        "No complete isolated promotion windows were found. "
        "Relax the event-window settings only after checking the data."
    )

raw_event_windows = pd.concat(
    event_window_rows,
    ignore_index=True,
)
isolated_events = pd.DataFrame(event_metadata_rows)

# ------------------------------------------------------------------
# Load the already-fitted rolling PPML models and their prediction
# context. No model is refitted in this notebook.
# ------------------------------------------------------------------
rolling_manifest = pd.read_pickle(ROLLING_MANIFEST_PATH)
prediction_context = pd.read_pickle(PREDICTION_CONTEXT_PATH)

for identifier in ["store_upc", "upc"]:
    prediction_context[identifier] = normalize_identifier(
        prediction_context[identifier]
    )

prediction_context["week"] = pd.to_numeric(
    prediction_context["week"],
    errors="raise",
).astype(int)

context_columns = [
    "store_upc",
    "upc",
    "week",
    "calendar_month",
    "scaled_time_trend",
    "thanksgiving_week",
    "christmas_week",
    "new_year_week",
    "easter_week",
    "log1p_lag_move",
    "log1p_lag_move_mean_4",
    "price_imputed_indicator",
]

missing_context_columns = set(context_columns).difference(
    prediction_context.columns
)
if missing_context_columns:
    raise ValueError(
        "Prediction context is missing required columns: "
        f"{sorted(missing_context_columns)}"
    )

event_windows = raw_event_windows.merge(
    prediction_context[context_columns],
    on=["store_upc", "upc", "week"],
    how="inner",
    validate="many_to_one",
)

# Attach the rolling model block that was used for each forecast week.
event_windows["model_block"] = pd.Series(
    pd.NA,
    index=event_windows.index,
    dtype="Int64",
)

for manifest_row in rolling_manifest.itertuples(index=False):
    block_mask = event_windows["week"].between(
        int(manifest_row.first_forecast_week),
        int(manifest_row.last_forecast_week),
    )
    event_windows.loc[
        block_mask,
        "model_block",
    ] = int(manifest_row.block_number)

event_windows = event_windows.dropna(
    subset=["model_block"]
).copy()
event_windows["model_block"] = (
    event_windows["model_block"].astype(int)
)

# Keep only events for which the full window is evaluable using saved
# out-of-sample rolling-model artifacts.
expected_window_length = len(relative_weeks)

event_completeness = (
    event_windows.groupby("event_id", observed=True)
    .agg(
        rows=("relative_week", "size"),
        distinct_weeks=("relative_week", "nunique"),
    )
)

complete_event_ids = event_completeness.index[
    event_completeness["rows"].eq(expected_window_length)
    & event_completeness["distinct_weeks"].eq(expected_window_length)
]

event_windows = event_windows.loc[
    event_windows["event_id"].isin(complete_event_ids)
].copy()

isolated_events = isolated_events.loc[
    isolated_events["event_id"].isin(complete_event_ids)
].copy()

if event_windows.empty:
    raise RuntimeError(
        "No isolated promotion events have complete rolling-PPML "
        "prediction context. Check that notebook 02 saved the rolling "
        "models and prediction context."
    )

# ------------------------------------------------------------------
# Predict a no-promotion baseline at the regular price.
# Promotion and post-promotion indicators are set to zero, while all
# other information remains as it was available at the forecast date.
# ------------------------------------------------------------------
event_windows["baseline_mu_hat"] = np.nan

manifest_by_block = (
    rolling_manifest
    .set_index("block_number")
    .sort_index()
)

for block_number, block_rows in event_windows.groupby(
    "model_block",
    observed=True,
):
    if block_number not in manifest_by_block.index:
        raise KeyError(
            f"Rolling-model block {block_number} is absent from the manifest."
        )

    manifest_row = manifest_by_block.loc[block_number]
    model_path = (
        ROLLING_MODEL_DIR / manifest_row["model_filename"]
    )

    if not model_path.exists():
        raise FileNotFoundError(
            f"Saved rolling PPML model not found: {model_path}"
        )

    fitted_model = load_pickle(model_path)
    counterfactual = block_rows.copy()

    counterfactual["promotion_indicator"] = 0
    counterfactual["post_promotion_indicator"] = 0
    counterfactual["discount_depth_model"] = 0.0
    counterfactual["discount_depth_sq"] = 0.0

    baseline_price = counterfactual["regular_price"].where(
        counterfactual["regular_price"].gt(0),
        counterfactual["model_unit_price"],
    )

    if baseline_price.isna().any() or ~baseline_price.gt(0).all():
        raise ValueError(
            f"Invalid baseline prices in rolling-model block {block_number}."
        )

    counterfactual["log_price_model"] = np.log(baseline_price)
    counterfactual["log_price_spline"] = (
        counterfactual["log_price_model"].clip(
            float(manifest_row["spline_lower"]),
            float(manifest_row["spline_upper"]),
        )
    )

    for categorical_column in [
        "store_upc",
        "upc",
        "calendar_month",
    ]:
        counterfactual[categorical_column] = (
            counterfactual[categorical_column]
            .astype("string")
        )

    baseline_prediction = np.asarray(
        fitted_model.predict(counterfactual),
        dtype=float,
    )

    if (
        not np.isfinite(baseline_prediction).all()
        or (baseline_prediction <= 0).any()
    ):
        raise ValueError(
            f"Invalid no-promotion predictions in block {block_number}."
        )

    event_windows.loc[
        block_rows.index,
        "baseline_mu_hat",
    ] = baseline_prediction

if event_windows["baseline_mu_hat"].isna().any():
    raise RuntimeError(
        "Some event rows did not receive a no-promotion baseline prediction."
    )

event_windows["residualized_sales_index"] = (
    100.0
    * event_windows["move"]
    / event_windows["baseline_mu_hat"]
)

# Raw pre-period-normalized path retained only as a diagnostic.
raw_event_path = (
    event_windows.groupby("relative_week", observed=True)
    .agg(
        mean_sales_index=("raw_sales_index", "mean"),
        n_events=("event_id", "nunique"),
    )
    .reset_index()
)

# ------------------------------------------------------------------
# Aggregate observed sales relative to the conditional PPML baseline.
# The point estimate is a ratio of sums. Confidence intervals resample
# complete store-product panels, preserving dependence among repeated
# promotion episodes from the same panel.
# ------------------------------------------------------------------
required_event_columns = {
    "event_id",
    "store_upc",
    "relative_week",
    "move",
    "baseline_mu_hat",
}

missing_event_columns = required_event_columns.difference(
    event_windows.columns
)
if missing_event_columns:
    raise RuntimeError(
        "The rolling-PPML baseline step did not finish correctly. "
        f"Missing columns: {sorted(missing_event_columns)}"
    )

if event_windows["baseline_mu_hat"].isna().any():
    missing_predictions = int(
        event_windows["baseline_mu_hat"].isna().sum()
    )
    raise RuntimeError(
        f"{missing_predictions:,} event rows have missing PPML "
        "baseline predictions."
    )

event_panel_lookup = (
    event_windows[
        ["event_id", "store_upc"]
    ]
    .drop_duplicates()
    .set_index("event_id")["store_upc"]
)

observed_matrix = (
    event_windows.pivot(
        index="event_id",
        columns="relative_week",
        values="move",
    )
    .reindex(columns=relative_weeks)
    .sort_index()
)

baseline_matrix = (
    event_windows.pivot(
        index="event_id",
        columns="relative_week",
        values="baseline_mu_hat",
    )
    .reindex(columns=relative_weeks)
    .sort_index()
)

if observed_matrix.isna().any().any():
    raise RuntimeError("Observed event matrix is incomplete.")

if baseline_matrix.isna().any().any():
    raise RuntimeError("Baseline event matrix is incomplete.")

event_panel_lookup = event_panel_lookup.reindex(
    observed_matrix.index
)

if event_panel_lookup.isna().any():
    raise RuntimeError(
        "Some event IDs could not be matched to store-product panels."
    )

observed_array = observed_matrix.to_numpy(dtype=float)
baseline_array = baseline_matrix.to_numpy(dtype=float)

point_estimate = (
    100.0
    * observed_array.sum(axis=0)
    / baseline_array.sum(axis=0)
)

panel_labels = event_panel_lookup.to_numpy()
unique_panels = pd.unique(panel_labels)

panel_to_event_positions = {
    panel: np.flatnonzero(panel_labels == panel)
    for panel in unique_panels
}

rng = np.random.default_rng(EVENT_BOOTSTRAP_SEED)

bootstrap_estimates = np.empty(
    (
        EVENT_BOOTSTRAP_REPLICATIONS,
        len(relative_weeks),
    ),
    dtype=float,
)

for replication in range(EVENT_BOOTSTRAP_REPLICATIONS):
    sampled_panels = rng.choice(
        unique_panels,
        size=len(unique_panels),
        replace=True,
    )

    sampled_event_positions = np.concatenate(
        [
            panel_to_event_positions[panel]
            for panel in sampled_panels
        ]
    )

    bootstrap_estimates[replication] = (
        100.0
        * observed_array[sampled_event_positions].sum(axis=0)
        / baseline_array[sampled_event_positions].sum(axis=0)
    )

residualized_event_path = pd.DataFrame(
    {
        "relative_week": relative_weeks,
        "sales_index": point_estimate,
        "ci_lower": np.quantile(
            bootstrap_estimates,
            0.025,
            axis=0,
        ),
        "ci_upper": np.quantile(
            bootstrap_estimates,
            0.975,
            axis=0,
        ),
        "n_events": observed_matrix.shape[0],
        "n_panels": len(unique_panels),
    }
)

print(
    "Complete isolated events with rolling-PPML baselines:",
    f"{observed_matrix.shape[0]:,}",
)
print(
    "Store-product panels represented:",
    f"{len(unique_panels):,}",
)
display(residualized_event_path)


## 8. Create paper-ready tables

In [ ]:
product_lookup = pd.read_parquet(PRODUCT_LOOKUP_PATH)
product_lookup["upc"] = normalize_identifier(product_lookup["upc"])

product_episode_summary = (
    promotion_episodes.groupby("upc", observed=True)
    .agg(
        promotion_episodes=("episode_number", "size"),
        single_week_episodes=(
            "promotion_weeks",
            lambda x: int((x == 1).sum()),
        ),
    )
    .reset_index()
)

isolated_episode_summary = (
    isolated_events.groupby("upc", observed=True)
    .agg(
        isolated_complete_episodes=("event_id", "nunique")
    )
    .reset_index()
)

product_descriptives = (
    selected_sample.groupby("upc", observed=True)
    .agg(
        observations=("move", "size"),
        stores=("store", "nunique"),
        panels=("store_upc", "nunique"),
        weeks=("week", "nunique"),
        total_units=("move", "sum"),
        median_weekly_sales=("move", "median"),
        promotion_share=("promotion_indicator", "mean"),
        median_promotion_depth=(
            "discount_depth",
            lambda x: x[
                selected_sample.loc[x.index, "promotion_indicator"].eq(1)
            ].median(),
        ),
        observed_price_share=(
            "price_imputed",
            lambda x: 1.0 - x.mean(),
        ),
    )
    .reset_index()
)

selected_product_audit = product_selection.loc[
    product_selection["upc"].isin(selected_upcs),
    [
        "upc",
        "train_units",
        "train_weeks",
        "train_stores",
        "train_promotion_rows",
        "train_distinct_prices",
        "train_relative_price_range",
        "train_imputed_share",
    ],
].copy()

product_table = (
    pd.DataFrame({"upc": selected_upcs})
    .merge(product_lookup, on="upc", how="left", validate="one_to_one")
    .merge(
        product_descriptives,
        on="upc",
        how="left",
        validate="one_to_one",
    )
    .merge(
        product_episode_summary,
        on="upc",
        how="left",
        validate="one_to_one",
    )
    .merge(
        isolated_episode_summary,
        on="upc",
        how="left",
        validate="one_to_one",
    )
    .merge(
        admissible_depth_strings,
        on="upc",
        how="left",
        validate="one_to_one",
    )
    .merge(
        selected_product_audit,
        on="upc",
        how="left",
        validate="one_to_one",
    )
)

product_table["product"] = (
    product_table["descrip"]
    .astype("string")
    .fillna("Unknown product")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
product_table["package_size"] = (
    product_table["size"]
    .astype("string")
    .fillna("")
    .str.strip()
)
product_table["supported_depths"] = (
    product_table["supported_depths"].fillna("None")
)
for count_column in [
    "promotion_episodes",
    "single_week_episodes",
    "isolated_complete_episodes",
]:
    product_table[count_column] = (
        product_table[count_column].fillna(0).astype(int)
    )

product_table = (
    product_table.sort_values(
        "train_units",
        ascending=False,
    )
    .reset_index(drop=True)
)

paper_product_table = product_table[
    [
        "product",
        "package_size",
        "stores",
        "median_weekly_sales",
        "promotion_share",
        "promotion_episodes",
        "isolated_complete_episodes",
        "supported_depths",
    ]
].copy()

paper_product_table = paper_product_table.rename(
    columns={
        "product": "Product",
        "package_size": "Size",
        "stores": "Stores",
        "median_weekly_sales": "Median weekly sales",
        "promotion_share": "Promotion share",
        "promotion_episodes": "Promotion episodes",
        "isolated_complete_episodes": "Isolated episodes",
        "supported_depths": "Supported depths (%)",
    }
)

paper_product_table["Median weekly sales"] = (
    paper_product_table["Median weekly sales"].round(1)
)
paper_product_table["Promotion share"] = (
    100.0 * paper_product_table["Promotion share"]
).map(lambda value: f"{value:.1f}%")

sample_summary = pd.Series(
    {
        "Rows": len(selected_sample),
        "Products": selected_sample["upc"].nunique(),
        "Stores": selected_sample["store"].nunique(),
        "Store-product panels": selected_sample["store_upc"].nunique(),
        "Weeks": selected_sample["week"].nunique(),
        "Start date": selected_sample["week_start"].min(),
        "End date": selected_sample["week_end"].max(),
        "Zero-sales share": selected_sample["move"].eq(0).mean(),
        "Promotion share": selected_sample[
            "promotion_indicator"
        ].mean(),
        "Imputed-price share": selected_sample[
            "price_imputed"
        ].mean(),
        "Promotion episodes": len(promotion_episodes),
        "Complete isolated episodes": len(isolated_events),
    },
    name="value",
)

descriptive_variables = {
    "Weekly unit sales": "move",
    "Unit price": "model_unit_price",
    "Regular price": "regular_price",
    "Discount depth": "discount_depth",
    "Observed gross margin": "gross_margin_pct_observed",
}

descriptive_rows = []
for label, column in descriptive_variables.items():
    values = pd.to_numeric(
        selected_sample[column],
        errors="coerce",
    ).dropna()

    if label == "Discount depth":
        values = values.loc[
            selected_sample.loc[
                values.index,
                "promotion_indicator",
            ].eq(1)
        ]

    descriptive_rows.append(
        {
            "Variable": label,
            "N": len(values),
            "Mean": values.mean(),
            "SD": values.std(),
            "P25": values.quantile(0.25),
            "Median": values.median(),
            "P75": values.quantile(0.75),
        }
    )

descriptive_table = pd.DataFrame(descriptive_rows)

display(paper_product_table)
display(sample_summary.to_frame())
display(descriptive_table)

## 9. Create the two-panel descriptive figure

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

pooled_depth_support = (
    support_source.groupby("depth_cluster", observed=True)
    .agg(
        promotion_observations=("move", "size"),
        products=("upc", "nunique"),
        panels=("store_upc", "nunique"),
    )
    .reset_index()
    .sort_values("depth_cluster")
)

depth_percent = (
    100.0 * pooled_depth_support["depth_cluster"]
)
total_promotion_observations = int(
    pooled_depth_support["promotion_observations"].sum()
)
event_count = int(
    residualized_event_path["n_events"].iloc[0]
)

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(11.5, 4.3),
    constrained_layout=True,
)

# ------------------------------------------------------------------
# Panel A: historical support by promotion-depth cluster
# ------------------------------------------------------------------
axes[0].bar(
    depth_percent,
    pooled_depth_support["promotion_observations"],
    width=4.0,
)
axes[0].set_xlabel("Promotion depth (%)")
axes[0].set_ylabel("Product-store-week observations")
axes[0].set_title("(a) Observed promotion depths")

minimum_depth = int(
    5 * np.floor(depth_percent.min() / 5)
)
maximum_depth = int(
    5 * np.ceil(depth_percent.max() / 5)
)

axes[0].set_xticks(
    np.arange(
        minimum_depth,
        maximum_depth + 1,
        5,
    )
)
axes[0].set_xlim(
    minimum_depth - 3,
    maximum_depth + 3,
)

axes[0].text(
    0.98,
    0.95,
    rf"$N={total_promotion_observations:,}$",
    transform=axes[0].transAxes,
    ha="right",
    va="top",
)

# ------------------------------------------------------------------
# Panel B: sales relative to a rolling-PPML no-promotion baseline
# ------------------------------------------------------------------
axes[1].plot(
    residualized_event_path["relative_week"],
    residualized_event_path["sales_index"],
    marker="o",
    linewidth=1.5,
)
axes[1].fill_between(
    residualized_event_path["relative_week"],
    residualized_event_path["ci_lower"],
    residualized_event_path["ci_upper"],
    alpha=0.20,
)
axes[1].axvline(
    0,
    linestyle="--",
    linewidth=1,
)
axes[1].axhline(
    100,
    linestyle=":",
    linewidth=1,
)
axes[1].set_xlabel("Week relative to promotion")
axes[1].set_ylabel(
    "Observed sales / no-promotion baseline (%)"
)
axes[1].set_title(
    "(b) Sales relative to no-promotion baseline"
)
axes[1].set_xticks(relative_weeks)

axes[1].text(
    0.02,
    0.95,
    rf"$N={event_count:,}$ episodes",
    transform=axes[1].transAxes,
    ha="left",
    va="top",
)

# ------------------------------------------------------------------
# Inset: post-promotion weeks on a readable scale
# ------------------------------------------------------------------
post_event_path = residualized_event_path.loc[
    residualized_event_path["relative_week"].between(
        1,
        POST_EVENT_WEEKS,
    )
].copy()

if not post_event_path.empty:
    inset = inset_axes(
        axes[1],
        width="43%",
        height="43%",
        loc="upper right",
        borderpad=1.1,
    )

    inset.plot(
        post_event_path["relative_week"],
        post_event_path["sales_index"],
        marker="o",
        linewidth=1.2,
    )
    inset.fill_between(
        post_event_path["relative_week"],
        post_event_path["ci_lower"],
        post_event_path["ci_upper"],
        alpha=0.20,
    )
    inset.axhline(
        100,
        linestyle=":",
        linewidth=0.9,
    )
    inset.set_title(
        "Weeks after promotion",
        fontsize=9,
    )
    inset.set_xticks(
        post_event_path["relative_week"]
    )
    inset.tick_params(
        axis="both",
        labelsize=8,
    )

    post_minimum = min(
        100.0,
        post_event_path["ci_lower"].min(),
    )
    post_maximum = max(
        100.0,
        post_event_path["ci_upper"].max(),
    )
    post_range = post_maximum - post_minimum
    post_margin = max(
        3.0,
        0.15 * post_range,
    )

    inset.set_ylim(
        post_minimum - post_margin,
        post_maximum + post_margin,
    )
    inset.set_xlim(
        0.75,
        POST_EVENT_WEEKS + 0.25,
    )

figure_png_path = (
    FIGURE_DIR
    / "paper_promotion_support_and_residualized_event_path.png"
)
figure_pdf_path = (
    FIGURE_DIR
    / "paper_promotion_support_and_residualized_event_path.pdf"
)

fig.savefig(
    figure_png_path,
    dpi=300,
    bbox_inches="tight",
)
fig.savefig(
    figure_pdf_path,
    bbox_inches="tight",
)
plt.show()

print("Saved:", figure_png_path)
print("Saved:", figure_pdf_path)

## 10. Save all paper outputs

In [ ]:
output_paths = {
    "paper product table CSV": (
        TABLE_DIR / "paper_product_characteristics.csv"
    ),
    "paper product table LaTeX": (
        TABLE_DIR / "paper_product_characteristics.tex"
    ),
    "sample summary CSV": (
        TABLE_DIR / "paper_sample_summary.csv"
    ),
    "descriptive statistics CSV": (
        TABLE_DIR / "paper_descriptive_statistics.csv"
    ),
    "descriptive statistics LaTeX": (
        TABLE_DIR / "paper_descriptive_statistics.tex"
    ),
    "split summary CSV": (
        TABLE_DIR / "paper_split_summary.csv"
    ),
    "promotion episodes CSV": (
        TABLE_DIR / "paper_promotion_episodes.csv"
    ),
    "isolated events CSV": (
        TABLE_DIR / "paper_isolated_promotion_events.csv"
    ),
    "residualized event path CSV": (
        TABLE_DIR
        / "paper_promotion_event_path_residualized.csv"
    ),
    "raw event path diagnostic CSV": (
        TABLE_DIR
        / "paper_promotion_event_path_raw_diagnostic.csv"
    ),
    "event windows with baseline Parquet": (
        PROCESSED_DIR
        / "paper_isolated_event_windows_with_ppml_baseline.parquet"
    ),
    "depth support CSV": (
        TABLE_DIR / "paper_promotion_depth_support.csv"
    ),
    "admissible depths CSV": (
        TABLE_DIR / "paper_admissible_promotion_depths.csv"
    ),
}

paper_product_table.to_csv(
    output_paths["paper product table CSV"],
    index=False,
)
paper_product_table.to_latex(
    output_paths["paper product table LaTeX"],
    index=False,
    escape=True,
    caption=(
        "Selected products and empirical promotion support."
    ),
    label="tab:product_characteristics",
)

sample_summary.to_csv(
    output_paths["sample summary CSV"],
    header=True,
)
descriptive_table.to_csv(
    output_paths["descriptive statistics CSV"],
    index=False,
)
descriptive_table.to_latex(
    output_paths["descriptive statistics LaTeX"],
    index=False,
    float_format="%.3f",
    caption="Descriptive statistics for the selected sample.",
    label="tab:descriptive_statistics",
)

split_summary.to_csv(
    output_paths["split summary CSV"],
    index=False,
)
promotion_episodes.to_csv(
    output_paths["promotion episodes CSV"],
    index=False,
)
isolated_events.to_csv(
    output_paths["isolated events CSV"],
    index=False,
)
residualized_event_path.to_csv(
    output_paths["residualized event path CSV"],
    index=False,
)
raw_event_path.to_csv(
    output_paths["raw event path diagnostic CSV"],
    index=False,
)
event_windows.to_parquet(
    output_paths["event windows with baseline Parquet"],
    index=False,
    engine="pyarrow",
    compression="zstd",
)
depth_support.to_csv(
    output_paths["depth support CSV"],
    index=False,
)
supported_candidates.to_csv(
    output_paths["admissible depths CSV"],
    index=False,
)

print("Saved paper outputs:")
for label, path in output_paths.items():
    print(f"  {label}: {path}")

## Notes for the paper

- The product and store choices are taken directly from the saved selection
  tables; this notebook does not select a new sample.
- The promotion-depth support calculation uses the training and calibration
  periods only. The test period remains excluded.
- The event-path panel uses the rolling PPML models already saved by notebook
  02; no demand model is refitted here.
- For each event week, the baseline prediction sets the promotion and
  post-promotion indicators to zero and evaluates demand at the regular price,
  while retaining the information available at that forecast date.
- Confidence intervals resample complete store-product panels, preserving
  dependence among repeated promotion episodes from the same panel.
- The event path remains descriptive and model-based rather than causal.
- If the final optimization notebook later saves an exact product-specific
  action grid, replace the transparent fallback in Section 6 with that saved
  grid so the table and optimization use identical depths.